# GRU On Isleme

**Bu notebook ne yapiyor?**
Ham GPS / nabiz / hiz zaman serilerini, `5_polarize.ipynb`'de uretilmis aerobik/anaerobik etiketleriyle birlestirip, bir GRU (Gated Recurrent Unit) modelinin dogrudan islebilecegi sayisal bir tensore donusturur.

**Pipeline'daki yeri nedir?**
`1_data_preprocess` -> `2_drop_columns` -> `3_haversine_savgol_filter` -> `4_Kmeans` (bu yaklasim daha sonra terk edildi) -> `5_polarize` (etiket uretimi: kisisel nabiz esigine gore aerobik/anaerobik siniflandirma) -> **`6_gru_preprocess`** -> `7_gru_tuner` (model kurulumu ve egitimi).

**Bu notebook somut olarak neyi cozuyor?**
Uc problem var: (1) etiketleri, hangi ham antrenmana ait olduklarini kaybetmeden dogru satirda eslestirmek, (2) ham GPS koordinatlarini modele verilebilir tek boyutlu bir hareket sinyaline indirgemek, (3) her antrenmani sabit boyutlu (4 kanal x 500 zaman adimi) bir diziye cevirip egitim/test setlerine bolup olceklemek. Asagidaki her bolum bu problemlerden birine karsilik geliyor.


In [2]:
import random, datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 1. Etiketleri yukle

**Bu etiketler nereden geliyor?**
`aerobik_anaerobik.csv`, `5_polarize.ipynb`'deki `run_zone_pipeline` fonksiyonu tarafindan uretildi. Her satir bir antrenmana karsilik geliyor ve iki hedef degisken tasiyor:

- `training_zone`: ikili etiket (`aerobik` / `anaerobik`), kisinin kendi nabiz tarihcesinden hesaplanan kisisellestirilmis bir esige gore belirleniyor.
- `anaerobic_time_frac`: 0 ile 1 arasinda surekli bir skor — antrenmanin ne kadarinin bu esigin ustunde gectigini gosteriyor (0.05, antrenmanin %95'inin aerobik, %5'inin anaerobik gectigi anlamina gelir). Bu skor ayrica saklaniyor cunku "easy run icinde kisa bir sprint" gibi karma antrenmanlari tek bir sinifa zorlamak bilgi kaybina yol aciyor.

**Neden `index_col=0`?**
Bu sutun, her etiketin ham dosyada (`endomondoHR_speed.csv`) kacinci satira ait oldugunu tutuyor. Bir sonraki adimda hizalamayi bu index uzerinden yapacagiz — `index_col=0` verilmezse bu bilgi sira numarasi gibi sikinti cikarmayan, ama aslinda kritik olan bir sutuna donusur.


In [3]:
labels_path = '/home/can/zero-to-ai-architect/runsight/aerobik_anaerobik.csv'

labels = pd.read_csv(labels_path, index_col=0)
print(f"labels: {labels.shape}")
labels.head()


labels: (69417, 7)


,training_zone,anaerobic_time_frac,duration_min,distance_km,speed_med,hr_med,hr_reserve_frac
1,anaerobik,0.246,70.166667,11.989848,10.042653,144.0,0.944979
2,aerobik,0.000,168.833333,31.786204,11.054558,128.0,0.486016
3,anaerobik,0.058,108.500000,23.969968,13.132261,131.0,0.706557
4,anaerobik,0.070,85.166667,16.415444,12.088747,129.0,0.807428
5,aerobik,0.010,50.666667,5.844217,7.018953,125.0,0.550206


## 2. Ham zaman serisini index uzerinden cek ve birlestir

**Neden tum 2GB'lik dosyayi okumuyoruz?**
`endomondoHR_speed.csv`'nin sadece etiketi olan satirlarina ihtiyacimiz var. `skiprows` parametresiyle, CSV'nin `i`. fiziksel satirinin `labels`'in `(i-1)`. index'ine karsilik gelip gelmedigine bakip, karsilik gelmeyen satirlari okuma asamasinda atliyoruz — bu, tum dosyayi belleğe alip sonra filtrelemekten cok daha az RAM kullaniyor.

**Neden `join()` ile birlestiriyoruz, sutun sirasina gore degil?**
`labels.join(speed_df)` index degerine gore eslestirme yapiyor — iki tablonun satir sirasi farkli olsa bile dogru sonucu veriyor. Sutun sirasina guvenen bir birlestirme (`pd.concat` gibi), iki dosyanin satir sirasi herhangi bir noktada kaysa sessizce yanlis satirlari birbirine baglar.

**Birlestirmeden hemen sonraki `assert`'ler ne icin var?**
Ikisi de hizalamanin dogru oldugunu dogruluyor: satir sayisi tutarli mi, birlestirme sonucunda hic NaN olustu mu. Bunlarin onemi su: index hizalama bir sekilde bozulursa (ornegin farkli bir QC calistirmasindan gelen bir index seti kullanilirsa), hicbir hata mesaji almadan yanlis etiketle egitilmis bir model elde edilebilir — sonuc calisir ama anlamsizdir. Assert'ler bu sessiz basarisizligi engelliyor.


In [4]:
PATH = 'endomondoHR_speed.csv'
# --- 1. Hangi orijinal satirlari cekecegimizi belirle ---
wanted_idx = set(labels.index)

# --- 2. Ham zaman serisi dosyasindan SADECE bu satirlari oku ---
# 2GB'lik dosyayi tamamen belleğe almamak icin skiprows kullaniyoruz:
# CSV'nin i. fiziksel satiri (header=0. satir) -> orijinal df index'i (i-1)
speed_df = pd.read_csv(
    PATH,
    index_col=0,
    skiprows=lambda i: i != 0 and (i - 1) not in wanted_idx,
)
print(f"speed_df (filtrelenmis): {speed_df.shape}")

# --- 3. Hizalamayi dogrula ---
assert len(speed_df) == len(labels), (
    f"Satir sayisi uyusmuyor: speed_df={len(speed_df)}, labels={len(labels)}"
)
missing = wanted_idx - set(speed_df.index)
assert not missing, f"speed_df'de eksik index'ler var: {sorted(missing)[:10]}"

# --- 4. Index uzerinden birlestir ---
df_merged = labels.join(speed_df, how="left")
n_nan = df_merged.isna().sum().sum()
assert n_nan == 0, f"Join sonrasi {n_nan} NaN degeri var, hizalama bozuk olabilir"

print(f"df_merged: {df_merged.shape[0]} satir, {df_merged.shape[1]} sutun")
df_merged.head()


speed_df (filtrelenmis): (69417, 10)
df_merged: 69417 satir, 17 sutun


,training_zone,anaerobic_time_frac,duration_min,distance_km,speed_med,hr_med,hr_reserve_frac,longitude,altitude,latitude,sport,id,heart_rate,gender,userId,timestamp,speed
1,anaerobik,0.246,70.166667,11.989848,10.042653,144.0,0.944979,[6.9144073 6.9142929 6.9141539 6.9140268 6.913...,[57.8 57.6 57. 56.4 55.8 55.2 54.4 53.4 52.6 ...,[52.2111711 52.2112631 52.2114064 52.2116083 5...,run,303565793,[ 60. 62. 92. 92. 132. 150. 150. 159. 159. ...,male,4969375,[1.39390853e+09 1.39390854e+09 1.39390855e+09 ...,[ 4.19558279 5.49069629 6.64752136 7.666057...
2,aerobik,0.000,168.833333,31.786204,11.054558,128.0,0.486016,[6.9141348 6.9145702 6.9151684 6.9158377 6.916...,[22.8 26.4 30.8 35.6 43. 48.4 49.8 49.4 50.2 ...,[52.2110297 52.2106325 52.2102453 52.2098332 5...,run,302666522,[ 77. 93. 107. 121. 118. 120. 120. 124. 124. ...,male,4969375,[1.39368793e+09 1.39368795e+09 1.39368797e+09 ...,[ 7.79265647 10.88124965 13.00812999 14.173297...
3,anaerobik,0.058,108.500000,23.969968,13.132261,131.0,0.706557,[6.8678543 6.8678634 6.8675429 6.8672183 6.867...,[35.4 35.2 34.6 34.2 35. 35.2 34.8 34.4 34.6 ...,[52.1936673 52.1934354 52.1931993 52.192873 5...,run,296982347,[ 75. 101. 116. 120. 124. 126. 127. 129. 126. ...,male,4969375,[1.39248016e+09 1.39248018e+09 1.39248019e+09 ...,[ 3.9874629 7.46274779 10.09532538 11.885195...
4,anaerobik,0.070,85.166667,16.415444,12.088747,129.0,0.807428,[6.9143328 6.9146396 6.9148949 6.9151568 6.915...,[63. 65.2 66. 66.2 65.8 65.8 67. 67. 66.8 ...,[52.2112195 52.2110264 52.2108135 52.2106013 5...,run,295890426,[ 58. 83. 112. 115. 117. 116. 141. 121. 120. ...,male,4969375,[1.39218043e+09 1.39218044e+09 1.39218045e+09 ...,[10.82019377 10.74224922 10.66821289 10.598084...
5,aerobik,0.010,50.666667,5.844217,7.018953,125.0,0.550206,[6.9133737 6.9132722 6.913217 6.9131066 6.913...,[61.4 56.6 53.2 49.6 46. 42.4 38.8 35.6 32.8 ...,[52.2111481 52.2111209 52.21119 52.2112981 5...,run,294163731,[ 56. 62. 75. 80. 80. 96. 107. 108. 113. ...,male,4969375,[1.39166354e+09 1.39166355e+09 1.39166356e+09 ...,[2.35707121 3.0496477 3.63514501 4.11356312 4...


## 3. Kanal hazirligi — hangi 4 sinyali kullaniyoruz ve neden

`df_merged`'daki dizi sutunlari hala metin (CSV'den boyle geldi) — once hepsini sayisal array'e ceviriyoruz. Veri seti onceki adimlarda (`1_data_preprocess`, `3_haversine_savgol_filter`) zaten temizlendigi icin burada ayrica bir NaN kontrolu / QC yapmiyoruz.

**GRU'ya hangi 4 zaman serisi kanalini veriyoruz?**

- `speed`: GPS'ten haversine formuluyle turetilip Savitzky-Golay filtresiyle yumusatilmis hiz — antrenmanin temposunu temsil ediyor.
- `heart_rate`: eforun dogrudan fizyolojik gostergesi.
- `altitude`: yukseklik degisimi. Bunu neden ekliyoruz? Ayni hizda giderken yokusta kalp atisi duzde olduğundan daha yuksek olur — sadece hiz ve nabza bakmak bu etkiyi ayirt edemez.
- `longitude`/`latitude` -> `latlon_pca`: ham GPS koordinatlarini dogrudan vermek yerine (bir sonraki bolumde aciklanan PCA adimiyla) hareketin net egilimini tek boyuta indirgeyip veriyoruz.

**Kisinin kisisel nabiz esigini (Karvonen HR reserve orani) neden modele vermiyoruz?**
Bilerek disarida birakildi. Amac, modelin etiketi "ezberlemesini" degil, ham kanallardan (hiz, nabiz, yukseklik, konum) efor oruntulerini gercekten ogrenmesini test etmek. Esik bilgisi verilseydi, model esigin uzerinde/altinda olup olmadigina bakmayi ogrenip asil ogrenmesi gereken zaman serisi oruntulerini atlayabilirdi.


In [5]:
# import sys
# sys.path.append('/home/can/zero-to-ai-architect/runsight')
import functions as fc

num_cols = ['longitude', 'altitude', 'latitude', 'heart_rate', 'timestamp', 'speed']

for col in num_cols:
    df_merged[col] = df_merged[col].apply(fc.parse_arr)

lengths = df_merged["speed"].apply(len)
print("dizi uzunluklari - min/max:", lengths.min(), lengths.max())


dizi uzunluklari - min/max: 500 500


## 4. Lat/lon -> tek boyut (PCA)

**Neden enlem/boylami dogrudan kullanmiyoruz?**
1 derece boylam, ekvatorda yaklasik 111 km'ye karsilik gelirken kutuplara yaklastikca kuculuyor — yani boylamin gercek mesafe karsiligi bulunulan enleme bagli (ayni gerekce `3_haversine_savgol_filter.ipynb`'de de gecti). Bu yuzden once her antrenmanin ilk noktasina gore lokal duz koordinata (metre) ceviriyoruz (esitrektangular yaklasim — kisa mesafeler icin haversine'e yeterince yakin bir sonuc verir).

**Sonra ne yapiyoruz?**
PCA(1) ile, donusturulmus (x, y) noktalarinin en cok varyans tasidigi tek eksene projekte ediyoruz. Sonuc, antrenman boyunca kisinin net olarak ne kadar ve hangi "yonde" (goreli olarak) hareket ettigini tasiyan tek bir sayi dizisi.

**Bu yaklasimin siniri nedir?**
PCA'nin isareti (yon) rastgele cikabilir — ayni rotayi kosan iki antrenmanin `latlon_pca` degerleri ters isaretli olabilir. Ayrica bu kanal, rotanin tam geometrisini degil, sadece net yer degistirme egilimini tasiyor — tam GPS izini yeniden kurmuyoruz. GRU'nun amaci harita cizmek degil, hareket yogunlugunu efor tahmininde kullanmak oldugu icin bu yeterli.


In [6]:
from sklearn.decomposition import PCA

R_EARTH = 6371000.0  # metre


def latlon_to_local_xy(lat, lon):
    lat0, lon0 = lat[0], lon[0]
    lat0_rad = np.radians(lat0)
    x = R_EARTH * np.radians(lon - lon0) * np.cos(lat0_rad)
    y = R_EARTH * np.radians(lat - lat0)
    return x, y


def pca_1d_track(lat, lon):
    x, y = latlon_to_local_xy(lat, lon)
    xy = np.column_stack([x, y])
    proj = PCA(n_components=1).fit_transform(xy).ravel()
    return proj


df_merged["latlon_pca"] = [
    pca_1d_track(lat, lon)
    for lat, lon in zip(df_merged["latitude"], df_merged["longitude"])
]


## 5. Kanallari istifle, hedefleri hazirla

**`X_seq` neden `(n_antrenman, 500, 4)` seklinde?**
4 kanali (speed, heart_rate, altitude, latlon_pca) tek bir 3 boyutlu diziye istifliyoruz. GRU katmani girdisini tam olarak bu formatta bekler: her ornek icin 500 zaman adimi x 4 ozellik.

**Neden iki ayri hedef (`y_binary`, `y_continuous`) tutuluyor?**
`y_binary`, `training_zone`'un 0/1'e cevrilmis hali — modelin egitildigi asil hedef bu. `y_continuous` ise `anaerobic_time_frac`; egitimde kullanilmiyor, ama model egitildikten sonra ciktisinin bu surekli skorla ne kadar ortustugune bakmak icin (baseline karsilastirmasi, `7_gru_tuner.ipynb`) saklaniyor.

**`np.bincount(y_binary)` ciktisi neyi gosteriyor?**
Iki sinifin (aerobik/anaerobik) veri setindeki dagilimini. Bu dagilim dengesizse (bir sinif digerinden belirgin sekilde fazlaysa), model bu cogunluk sinifina yanlilik gosterebilir — `7_gru_tuner.ipynb`'de `class_weight` kullanilmasinin gerekcesi tam olarak bu.


In [7]:
CHANNELS = ["speed", "heart_rate", "altitude", "latlon_pca"]

X_seq = np.stack(
    [np.column_stack([row[c] for c in CHANNELS]) for _, row in df_merged.iterrows()],
    axis=0,
)  # sekil: (n_ornek, 500, 4)

y_binary = (df_merged["training_zone"] == "anaerobik").astype(int).to_numpy()
y_continuous = df_merged["anaerobic_time_frac"].to_numpy()

print("X_seq shape:", X_seq.shape)
print("y_binary dagilimi (0=aerobik, 1=anaerobik):", np.bincount(y_binary))


X_seq shape: (69417, 500, 4)
y_binary dagilimi (0=aerobik, 1=anaerobik): [34899 34518]


## 6. Train/test split + kanal bazli olcekleme

**Split neden antrenman (workout) bazli, kullanici bazli degil?**
Veri setinde (~70 bin antrenman) tek bir kullanicinin agirlik basmasina izin verecek kadar az kullanici yok — yani ayni kisinin farkli antrenmanlarinin hem train hem test'e dusmesi, sonuclari carpitacak kadar buyuk bir risk degil. `stratify=y_binary` ile asil onemsenen sey saglaniyor: iki sinifin da train ve test'te ayni oranda temsil edilmesi.

**Olcekleme neden kanal bazli, tek bir scaler ile degil?**
speed (km/h), heart_rate (bpm), altitude (metre) ve latlon_pca (metre, ama farkli bir olcekte) tamamen farkli birimlerde. Hepsini tek bir scaler'a sokmak, en buyuk mutlak degerlere sahip kanalin digerlerini domine etmesine yol acardi. Bunun yerine her kanal kendi min/max degerine gore ayri ayri [0, 1] araligina cekiliyor.

**Neden `MinMaxScaler`, `StandardScaler` degil?**
GRU gibi tekrarlayan (recurrent) aglarda girdilerin sinirli bir araliga ([0, 1]) sikismis olmasi, ictteki `tanh`/`sigmoid` kapilarinin (gate) saglikli calismasina ve egitimin patlayici gradyanlara (exploding gradients) karsi daha kararli olmasina yardimci olur.

**Scaler neden sadece train setinde `fit` ediliyor?**
Test setinin istatistiklerini (min/max) olcekleme islemine karistirmak, modele test verisinden dolayli bir bilgi sizdirir (data leakage) — bu, modelin gercek dunyada hicbir zaman sahip olamayacagi bir avantaj olurdu. Bu yuzden `fit_transform` sadece train'de, test'e sadece `transform` uygulaniyor.


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler

X_train, X_test, y_train, y_test, yc_train, yc_test = train_test_split(
    X_seq, y_binary, y_continuous,
    test_size=0.2, random_state=46, stratify=y_binary,
)
print("egitim:", X_train.shape, "test:", X_test.shape)

n_channels = X_train.shape[2]
scalers = []
X_train_scaled = np.empty_like(X_train)
X_test_scaled = np.empty_like(X_test)

for c in range(n_channels):
    scaler = MinMaxScaler()
    X_train_scaled[:, :, c] = scaler.fit_transform(X_train[:, :, c])
    X_test_scaled[:, :, c] = scaler.transform(X_test[:, :, c])
    scalers.append(scaler)

print("olcekleme tamam. kanal sirasi:", CHANNELS)


egitim: (55533, 500, 4) test: (13884, 500, 4)
olcekleme tamam. kanal sirasi: ['speed', 'heart_rate', 'altitude', 'latlon_pca']


## On isleme nesnelerini kaydet

`scalers` ve kanal sirasi, ileride model deploy edilirken (serverless inference) ayni donusumu yeniden uygulayabilmek icin lazim — model dosyasindan ayri ama onunla birlikte saklanmali.


In [9]:
import pickle

with open('/home/can/zero-to-ai-architect/runsight/gru_preprocessing.pkl', 'wb') as f:
    pickle.dump({"channel_order": CHANNELS, "scalers": scalers}, f)

print("on isleme nesneleri kaydedildi.")


on isleme nesneleri kaydedildi.


## Egitim array'lerini diske kaydet

Onceki hucre sadece scaler'lari kaydediyordu — asil `X_train_scaled` / `X_test_scaled` / `y_train` / `y_test` array'leri bu kernel'de kaliyordu. 7_gru_tuner.ipynb'nin bu notebook'un kernel'ine ihtiyac duymadan, ayri bir kernel'de calisabilmesi icin array'leri de `float32` olarak (yer kaplamayi yariya indirir) diske yaziyoruz. Kaydettikten sonra bu notebook'un kernel'ini kapatip RAM'i bosaltabilirsin — 7. notebook hicbir seyi burdan miras almiyor, hepsini diskten okuyor.


In [10]:
np.savez(
    '/home/can/zero-to-ai-architect/runsight/gru_arrays.npz',
    X_train_scaled=X_train_scaled.astype(np.float32),
    X_test_scaled=X_test_scaled.astype(np.float32),
    y_train=y_train,
    y_test=y_test,
    yc_train=yc_train,
    yc_test=yc_test,
)
print("array'ler kaydedildi -> gru_arrays.npz")


array'ler kaydedildi -> gru_arrays.npz
